# ARC AGI 3 Single Game Ranker Diagnostic

This notebook is for diagnosing one public game, not for producing the final hidden-game checkpoint.

It filters search or collector trajectories to one game, builds candidate ranking examples, and trains a small ranker with an episode split. Use it to inspect whether the visual and source verified features are meaningful for that game before trusting a full multi-game run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

PROJECT_SOURCE_MODE = 'drive_dir'  # Default and recommended. Use 'drive_zip' only if you uploaded a project bundle zip.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3')
DRIVE_PROJECT_ZIP = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3/Local_Output/Colab_Bundles/arc_agi3_colab_bundle.zip')
DRIVE_INPUT_DATA_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data')
DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Training_Output')
LOCAL_WORKDIR = Path('/content/ARC Prize 2026 - ARC-AGI-3')
LOCAL_INPUT_DATA_BASE = Path('/content/ARC2026_AGI_3_Input_Data')
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_INPUT_DATA_BASE.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print('Project source mode:', PROJECT_SOURCE_MODE)
print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Drive project zip:', DRIVE_PROJECT_ZIP)
print('Drive input data base:', DRIVE_INPUT_DATA_BASE)
print('Drive output base:', DRIVE_OUTPUT_BASE)
print('Local workdir:', LOCAL_WORKDIR)
print('Local input data base:', LOCAL_INPUT_DATA_BASE)
print('Output root:', OUTPUT_ROOT)


In [ ]:
import shutil

if LOCAL_WORKDIR.exists():
    shutil.rmtree(LOCAL_WORKDIR)
LOCAL_WORKDIR.parent.mkdir(parents=True, exist_ok=True)

if PROJECT_SOURCE_MODE == 'drive_dir':
    get_ipython().system('rsync -a --delete --exclude .git --exclude .venv --exclude __pycache__ --exclude .DS_Store --exclude Local_Output --exclude Gif_Demo --exclude OpenLab_Backup_* "{}"/ "{}"/'.format(DRIVE_PROJECT_ROOT, LOCAL_WORKDIR))
elif PROJECT_SOURCE_MODE == 'drive_zip':
    if not DRIVE_PROJECT_ZIP.exists():
        raise FileNotFoundError(f'Project zip not found: {DRIVE_PROJECT_ZIP}')
    get_ipython().system('unzip -q "{}" -d /content'.format(DRIVE_PROJECT_ZIP))
else:
    raise ValueError(f'Unsupported PROJECT_SOURCE_MODE: {PROJECT_SOURCE_MODE}')

get_ipython().run_line_magic('cd', str(LOCAL_WORKDIR))


In [ ]:
import importlib.util, subprocess
from pathlib import Path

def run(cmd):
    print('>>>', cmd)
    subprocess.check_call(cmd, shell=True)

def run_streaming(cmd, log_path=None):
    print('>>>', cmd)
    handle = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        handle = log_path.open('a', encoding='utf-8')
    try:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            if handle is not None:
                handle.write(line)
                handle.flush()
        return_code = proc.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
    finally:
        if handle is not None:
            handle.close()

def has_module(name):
    return importlib.util.find_spec(name) is not None

run('python -m pip install -U pip wheel setuptools')

if has_module('torch'):
    print('torch already available, skipping torch/vision/audio install')
else:
    run('python -m pip install -U torch torchvision torchaudio')

if has_module('arc_agi') and has_module('arcengine'):
    print('arc_agi and arcengine already available, skipping install')
else:
    try:
        run('python -m pip install -U arc-agi==0.9.8 arcengine==0.9.3')
    except Exception:
        print('PyPI install failed, trying local wheels...')
        run('python -m pip install arc_agi_3_wheels/*.whl')

run('python - <<\'PY\'\nimport torch\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("device", torch.cuda.get_device_name(0))\nPY')


In [ ]:
# Configure one-game ranker diagnostics.
# Use source/search trajectory logs instead of raw human demos when possible.

SINGLE_GAME_ID = 'ar25'  # Change this to ar25, ls20, r11l, lp85, etc.
EPISODE_VAL_FRACTION = 0.2
RUN_TAG = f'single_game_ranker_{SINGLE_GAME_ID}'
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TAG / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_TRAJECTORY_GZ = DRIVE_INPUT_DATA_BASE / 'openlab_ranker_collect' / 'all_episodes.jsonl.gz'
if not SOURCE_TRAJECTORY_GZ.exists():
    raise FileNotFoundError(
        f'Source/search trajectory gzip not found: {SOURCE_TRAJECTORY_GZ}. Run src.collect_openlab first.'
    )

LOCAL_INPUT_DATA_BASE.mkdir(parents=True, exist_ok=True)
LOCAL_SINGLE_GAME_GZ = LOCAL_INPUT_DATA_BASE / f'single_game_source_{SINGLE_GAME_ID}.episodes.jsonl.gz'
RANKER_DATA_PATH = OUTPUT_ROOT / f'{SINGLE_GAME_ID}.ranker_examples.jsonl.gz'
RANKER_METADATA_PATH = OUTPUT_ROOT / f'{SINGLE_GAME_ID}.ranker_metadata.json'

RANKER_COORD_BUDGET = 32
RANKER_MAX_STEPS = 240
RANKER_MIN_POSITIVE_UTILITY = 0.05

print('Single game:', SINGLE_GAME_ID)
print('Episode val fraction:', EPISODE_VAL_FRACTION)
print('Output root:', OUTPUT_ROOT)
print('Source trajectory data:', SOURCE_TRAJECTORY_GZ)
print('Single-game trajectory path:', LOCAL_SINGLE_GAME_GZ)


In [ ]:
# Filter trajectories to one game, then build ranker examples.

filter_cmd = (
    f'python -m src.filter_episodes '
    f'--input "{SOURCE_TRAJECTORY_GZ}" '
    f'--output "{LOCAL_SINGLE_GAME_GZ}" '
    f'--games "{SINGLE_GAME_ID}"'
)

run_streaming(filter_cmd, OUTPUT_ROOT / 'filter_stdout.log')

build_ranker_cmd = (
    f'PYTHONUNBUFFERED=1 python -m src.build_ranker_dataset '
    f'--episodes "{LOCAL_SINGLE_GAME_GZ}" '
    f'--output "{RANKER_DATA_PATH}" '
    f'--metadata-output "{RANKER_METADATA_PATH}" '
    f'--coord-budget {RANKER_COORD_BUDGET} '
    f'--max-steps {RANKER_MAX_STEPS} '
    f'--min-positive-utility {RANKER_MIN_POSITIVE_UTILITY} '
    f'--progress-every 250'
)

run_streaming(build_ranker_cmd, OUTPUT_ROOT / 'build_ranker_dataset_stdout.log')


In [ ]:
# Train the one-game diagnostic ranker.
# This uses an episode split, so it checks within-game generalization only. It is weaker evidence than multi-game held out validation.

train_ranker_cmd = (
    f'PYTHONUNBUFFERED=1 python -m src.train_ranker '
    f'--data "{RANKER_DATA_PATH}" '
    f'--output-dir "{OUTPUT_ROOT}" '
    f'--split-mode episode '
    f'--epochs 12 '
    f'--batch-size 64 '
    f'--hidden-dim 96 '
    f'--dropout 0.10 '
    f'--lr 3e-4 '
    f'--weight-decay 1e-3 '
    f'--min-target {RANKER_MIN_POSITIVE_UTILITY} '
    f'--max-candidates 48'
)

run_streaming(train_ranker_cmd, OUTPUT_ROOT / 'train_ranker_stdout.log')


In [ ]:
# Inspect diagnostic metrics.

import json
import pandas as pd

metrics_path = OUTPUT_ROOT / 'metrics.csv'
summary_path = OUTPUT_ROOT / 'summary.json'
metadata_path = RANKER_METADATA_PATH

display(pd.read_csv(metrics_path).tail())
print('Best ranker checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'best_ranker.pth')
print('Last ranker checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'last_ranker.pth')
print('
Summary JSON:')
print(json.dumps(json.loads(summary_path.read_text()), indent=2)[:4000])
print('
Dataset metadata:')
print(json.dumps(json.loads(metadata_path.read_text()), indent=2)[:4000])
